# Project: Wildfire Mapping

Goal: Build an html side with an interactive map where the user can see the recents wildfires. Provide the user with information of phisical size, duration, intensity, etc. of the wildfires. Use pop-ups and tooltips to make the map interactive and structured for the users. The map should be for public users which are interesting in wildfires.

Tasks:
1. Load the api
2. Extract the data which is used to locate the wildfire (VIIRS_SNPP_NRT)
3. Explore the data
4. Clean the data (if needed)
5. Check wildfires for different properties
6. Visulaization of the wildfires
7. Provide additional information about the wildfires. 
8. Make the map interactive

In [9]:
# import all libraries
import requests
import pandas as pd
import io
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from datetime import datetime
from shapely.geometry import MultiPoint

In [3]:
# Building the api url 

# get api key
api_key = "47778cf594ae276d2b7dfc098596de2a"
# define source
api_source = "VIIRS_SNPP_NRT" # or change ot to VIIRS_SNPP_SP?
# define area coordinates
api_area_coordinates = "world"
# day range. Days going back from today
api_day_range = 5
# build api url with api key
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{api_key}/{api_source}/{api_area_coordinates}/{api_day_range}"

In [4]:
# load the data form the api
response = requests.get(api_url)

# check if api import was successfull
if response.status_code == 200:
    print("API request successfull")

    # get the data as a csv
    data_csv = response.text
    # create a dataframe
    data_df = pd.read_csv(io.StringIO(data_csv))

else:
    print(f"Request failed. Status code: {response.status_code}")

API request successfull


In [15]:
# converting the data data frame into a geo-data frame
wildfire_gdf = gpd.GeoDataFrame(data_df, geometry=gpd.points_from_xy(data_df["longitude"], data_df["latitude"]), crs=4326)

# convert the date column into a date datetime type
wildfire_gdf["acq_date"] = pd.to_datetime(wildfire_gdf["acq_date"], format="%Y-%m-%d")

# preject to calculate in meters
wildfire_gdf = wildfire_gdf.to_crs(epsg=3857)
# join nearby spatial points according to the same fire (most likly the same)
wildfire_gdf["geometry_buffer"] = wildfire_gdf.geometry.buffer(751) # buffer of 751 meters. resolution of the data is 350mx750m
# join fires within the buffer
joined_wildfires = gpd.sjoin(
    wildfire_gdf[["acq_date", "geometry"]],
    wildfire_gdf[["acq_date", "geometry_buffer"]].set_geometry("geometry_buffer"),
    how="left", # make sure to keep all points
    predicate="within"
)

joined_wildfires = joined_wildfires[joined_wildfires["acq_date_left"] == joined_wildfires["acq_date_right"]] # Keep only fires with the same registration date
# aggregate into clusters
wildfire_clusters = (
    joined_wildfires.groupby("index_right").agg(
        acq_date=("acq_date_left", "first"),
        count=("acq_date_left", "count"),
        geometry=("geometry", lambda geoms: MultiPoint(list(geoms)).centroid)
    )
    .reset_index(drop=True)
)

# Build the cleaned aggregated wildfire data
wildfire_clusterd = gpd.GeoDataFrame(wildfire_clusters, geometry="geometry", crs=3857).to_crs(epsg=4326)


In [16]:
# verify transformation to geo data frame
display(wildfire_gdf.head(5))
display(wildfire_clusterd.head(5))
display(wildfire_clusterd.info)
display(wildfire_clusterd.dtypes)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,geometry,geometry_buffer
0,54.84750,56.10764,295.84,0.56,0.69,2026-05-11,7,N,VIIRS,n,2.0NRT,271.35,1.34,N,POINT (6245873.914 7332325.048),"POLYGON ((6246624.914 7332325.048, 6246621.298..."
1,55.56430,51.93401,295.12,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,270.48,0.51,N,POINT (5781267.548 7472164.569),"POLYGON ((5782018.548 7472164.569, 5782014.932..."
2,55.59990,51.90491,296.59,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,274.84,0.63,N,POINT (5778028.151 7479175.893),"POLYGON ((5778779.151 7479175.893, 5778775.535..."
3,55.60163,51.89975,295.72,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,274.97,0.63,N,POINT (5777453.742 7479516.774),"POLYGON ((5778204.742 7479516.774, 5778201.126..."
4,55.61058,51.93056,295.52,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,274.78,1.00,N,POINT (5780883.496 7481280.531),"POLYGON ((5781634.496 7481280.531, 5781630.88 ..."


,acq_date,count,geometry
0,2026-05-11,2,POINT (56.10817 54.84875)
1,2026-05-11,1,POINT (51.93401 55.5643)
2,2026-05-11,2,POINT (51.90233 55.60077)
3,2026-05-11,2,POINT (51.90233 55.60077)
4,2026-05-11,1,POINT (51.93056 55.61058)


<bound method DataFrame.info of          acq_date  count                   geometry
0      2026-05-11      2  POINT (56.10817 54.84875)
1      2026-05-11      1   POINT (51.93401 55.5643)
2      2026-05-11      2  POINT (51.90233 55.60077)
3      2026-05-11      2  POINT (51.90233 55.60077)
4      2026-05-11      1  POINT (51.93056 55.61058)
...           ...    ...                        ...
148939 2026-05-15      1     POINT (6.35805 50.085)
148940 2026-05-15      1    POINT (8.18418 50.4954)
148941 2026-05-15      1  POINT (25.07224 50.56683)
148942 2026-05-15      1   POINT (9.22623 50.90878)
148943 2026-05-15      1  POINT (24.83804 51.71843)

[148944 rows x 3 columns]>

acq_date    datetime64[us]
count                int64
geometry          geometry
dtype: object

In [19]:
# initalize Folium back ground map
background_map = folium.Map(
    location=[0, 0], # start zoom at latitude and longitude 0
    zoom_start=2, # shows the whole word at the start
    tiles="CartoDB DarkMatter", # Dark basmap
    control_scale=True # Add scalebar
)

cluster_fire = MarkerCluster(name="Recent Fires").add_to(background_map)
# build markers for the wildfires
for idx, row in wildfire_clusterd.iterrows():
    lat = row.geometry.y # extract latitude out of the geometry column 
    lon = row.geometry.x # eextract longitude out of the geometry column
    count = row["count"] # counting how many fires are aggregated

    # Formating the Tooltip. Date of the fire registation
    fire_start = row["acq_date"].date()
    tooltip = f"Date: {fire_start} | Detections: {count}"

    # Define Marker color deoending the count of fires
    if count == 1:
        color =  "orange"
    elif count <= 5:
        color = "red"
    else:
        color = "darkred"

    # create Marker of the fire locations
    folium.Marker(
        location=[lat, lon],
        tooltip=tooltip,
        icon=folium.Icon(color=color, icon="fire", prefix="fa")
    ).add_to(cluster_fire)

folium.LayerControl().add_to(background_map)
#background_map
background_map.save("../outputs/map.html")

datetime.date(2026, 5, 15)